# Retirement Planner

A self-contained model for projecting your retirement portfolio,
estimating *when* you can retire, and stress-testing that plan against
market variability.

Use the **sidebar** to adjust your assumptions. The deterministic projection
and earliest retirement age update live as you change any input.
Click **Run Monte Carlo** when you want the randomised stress-test results.

**Important modeling choice — everything is in *today's dollars* (real terms).**
Enter your expenses, savings, contributions, and Social Security in today's
purchasing power, and enter *real* (inflation-adjusted) rates of return
(e.g. if you expect 7% nominal returns and 2.5% inflation, use ~4.5%).
This avoids tracking a separate inflation rate while keeping every number
intuitive — "70k/year" always means 70k of *today's* buying power,
no matter how far in the future.

This is a planning aid, not financial advice. See the **Notes & Caveats**
section at the end for the simplifications baked into this model.

In [6]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import Layout, AppLayout
from IPython.display import display

matplotlib.rcParams["figure.dpi"] = 110
pd.options.display.float_format = lambda x: f"{x:,.0f}"

In [7]:
def project_portfolio(a, retirement_age, return_sequence=None):
    """
    Simulate the portfolio from current_age to life_expectancy.

    Parameters
    ----------
    a : dict
        The assumptions dictionary.
    retirement_age : int
        Age at which contributions stop and withdrawals begin.
    return_sequence : list[float], optional
        One real annual return per simulated year. If omitted, uses the
        constant pre_retirement_return / post_retirement_return from `a`.

    Returns
    -------
    pandas.DataFrame with one row per year of the simulation.
    """
    years = a["life_expectancy"] - a["current_age"]
    balance = a["current_savings"]
    contribution = a["annual_contribution"]
    rows = []

    for i in range(years):
        age = a["current_age"] + i
        retired = age >= retirement_age

        if return_sequence is not None:
            r = return_sequence[i]
        else:
            r = a["post_retirement_return"] if retired else a["pre_retirement_return"]

        ss_income = a["social_security_monthly"] * 12 if age >= a["social_security_start_age"] else 0
        pension_income = (
            a["pension_monthly"] * 12
            if a["pension_start_age"] is not None and age >= a["pension_start_age"]
            else 0
        )

        balance_start = balance

        if retired:
            withdrawal = max(a["annual_expenses"] - ss_income - pension_income, 0)
            balance -= withdrawal
            contrib_this_year = 0
        else:
            withdrawal = 0
            balance += contribution
            contrib_this_year = contribution
            contribution *= (1 + a["contribution_growth_rate"])

        growth = balance * r
        balance += growth

        rows.append({
            "age": age,
            "retired": retired,
            "balance_start": balance_start,
            "contribution": contrib_this_year,
            "ss_income": ss_income,
            "pension_income": pension_income,
            "withdrawal": withdrawal,
            "growth": growth,
            "balance_end": balance,
        })

    return pd.DataFrame(rows)


def find_earliest_retirement_age(a, age_range=None):
    if age_range is None:
        age_range = range(a["current_age"] + 1, a["life_expectancy"])

    for age in age_range:
        df = project_portfolio(a, age)
        if (df["balance_end"] >= 0).all():
            return age, df

    return None, None


def monte_carlo_success(a, retirement_age, n_sims=1000, seed=42):
    """
    Run n_sims randomized projections for a given retirement age.

    Returns
    -------
    success_rate : float
        Fraction of simulations where the portfolio never goes negative.
    ending_balances : list[float]
        Final balance (at life expectancy) for each simulation.
    """
    rng = np.random.default_rng(seed)
    years = a["life_expectancy"] - a["current_age"]
    successes = 0
    ending_balances = []

    for _ in range(n_sims):
        returns = []
        for i in range(years):
            age = a["current_age"] + i
            mean_return = a["post_retirement_return"] if age >= retirement_age else a["pre_retirement_return"]
            returns.append(rng.normal(mean_return, a["return_volatility"]))

        df = project_portfolio(a, retirement_age, return_sequence=returns)
        ending_balances.append(df["balance_end"].iloc[-1])
        if (df["balance_end"] >= 0).all():
            successes += 1

    return successes / n_sims, ending_balances

def mc_success_grid(a, ages, balances, n_sims=300, seed=42,
                    inflation_rate=0.0, base_age=None):
    """
    Vectorised success-rate grid over retirement ages × starting balances.
    Each cell answers: "if I retire at `age` with `balance`, what fraction
    of simulations survive to life expectancy?"
    When inflation_rate > 0 and base_age is set, `balances` are treated as
    nominal dollars and converted to real for each age column.
    All years use post_retirement_return (person is already retired).
    """
    rng = np.random.default_rng(seed)
    grid = np.zeros((len(balances), len(ages)))

    for j, age in enumerate(ages):
        years = a["life_expectancy"] - age
        if years <= 0:
            grid[:, j] = 1.0
            continue
        ret = rng.normal(a["post_retirement_return"], a["return_volatility"], (n_sims, years))

        # Net withdrawal per year (SS/pension offsets applied)
        withdrawals = np.zeros(years)
        for yr in range(years):
            curr_age = age + yr
            ss = a["social_security_monthly"] * 12 if curr_age >= a["social_security_start_age"] else 0
            pension = (
                a["pension_monthly"] * 12
                if a["pension_start_age"] is not None and curr_age >= a["pension_start_age"]
                else 0
            )
            withdrawals[yr] = max(a["annual_expenses"] - ss - pension, 0)

        if inflation_rate > 0 and base_age is not None:
            real_bals = [b / (1 + inflation_rate) ** (age - base_age) for b in balances]
        else:
            real_bals = balances

        for i, bal in enumerate(real_bals):
            b = np.full(n_sims, float(bal))
            alive = np.ones(n_sims, dtype=bool)
            for yr in range(years):
                b -= withdrawals[yr]
                b *= (1 + ret[:, yr])
                alive &= (b >= 0)
            grid[i, j] = alive.mean()

    return grid

In [8]:
_style = {"description_width": "170px"}
_layout = Layout(width="280px")

# --- Personal ---
w_current_age = widgets.BoundedIntText(
    value=56, min=18, max=90,
    description="Current age:", style=_style, layout=_layout)
w_target_retirement_age = widgets.BoundedIntText(
    value=65, min=40, max=80,
    description="Target ret. age:", style=_style, layout=_layout)
w_life_expectancy = widgets.BoundedIntText(
    value=90, min=70, max=110,
    description="Life expectancy:", style=_style, layout=_layout)
# --- Savings ---
w_current_savings = widgets.BoundedFloatText(
    value=250, min=0, max=1e6, step=10,
    description="Current savings ($k):", style=_style, layout=_layout)
w_annual_contribution = widgets.BoundedFloatText(
    value=15, min=0, max=500, step=1,
    description="Annual contribution ($k):", style=_style, layout=_layout)
w_contribution_growth_rate = widgets.BoundedFloatText(
    value=2.0, min=0, max=10.0, step=0.5,
    description="Contribution growth (%):", style=_style, layout=_layout)

# --- Returns ---
w_inflation_rate = widgets.BoundedFloatText(
    value=2.5, min=0, max=10.0, step=0.5,
    description="Inflation (%):", style=_style, layout=_layout)
w_pre_retirement_return = widgets.BoundedFloatText(
    value=7.0, min=0, max=20.0, step=0.5,
    description="Pre-ret. nominal (%):", style=_style, layout=_layout)
w_post_retirement_return = widgets.BoundedFloatText(
    value=5.0, min=0, max=15.0, step=0.5,
    description="Post-ret. nominal (%):", style=_style, layout=_layout)
w_return_volatility = widgets.BoundedFloatText(
    value=12.0, min=0, max=30.0, step=0.5,
    description="Return volatility (%):", style=_style, layout=_layout)

# --- Spending / Social Security ---
w_annual_expenses = widgets.BoundedFloatText(
    value=60, min=0, max=1e4, step=1,
    description="Annual expenses ($k):", style=_style, layout=_layout)
w_social_security_monthly = widgets.BoundedFloatText(
    value=1.8, min=0, max=50, step=0.1,
    description="SS monthly ($k):", style=_style, layout=_layout)
w_social_security_start_age = widgets.RadioButtons(
    options=[62, 65, 67, 70], value=67,
    description="SS start age:", style=_style, layout=_layout)

# --- Pension (gated) ---
w_has_pension = widgets.ToggleButton(
    value=False, description="Has pension?",
    button_style="", layout=_layout)
w_pension_monthly = widgets.BoundedFloatText(
    value=0, min=0, max=50, step=0.1,
    description="Pension monthly ($k):", style=_style, layout=_layout)
w_pension_start_age = widgets.BoundedIntText(
    value=60, min=40, max=80,
    description="Pension start age:", style=_style, layout=_layout)
pension_box = widgets.VBox([w_pension_monthly, w_pension_start_age])
pension_box.layout.display = "none"

# --- Monte Carlo controls ---
w_n_sims = widgets.RadioButtons(
    options=[500, 1000, 2000, 5000], value=1000,
    description="# simulations:", style=_style, layout=_layout)
run_mc_button = widgets.Button(
    description="Run Monte Carlo", button_style="primary", layout=_layout)
run_grid_button = widgets.Button(
    description="Run Balance Grid", button_style="info", layout=_layout)


def _toggle_pension(change=None):
    pension_box.layout.display = "" if w_has_pension.value else "none"


w_has_pension.observe(_toggle_pension, names="value")

def build_assumptions_from_widgets():
    return {
        "current_age": w_current_age.value,
        "life_expectancy": w_life_expectancy.value,
        "current_savings": w_current_savings.value * 1_000,
        "annual_contribution": w_annual_contribution.value * 1_000,
        "contribution_growth_rate": w_contribution_growth_rate.value / 100,
        "pre_retirement_return": (1 + w_pre_retirement_return.value / 100) / (1 + w_inflation_rate.value / 100) - 1,
        "post_retirement_return": (1 + w_post_retirement_return.value / 100) / (1 + w_inflation_rate.value / 100) - 1,
        "return_volatility": w_return_volatility.value / 100,
        "annual_expenses": w_annual_expenses.value * 1_000,
        "social_security_monthly": w_social_security_monthly.value * 1_000,
        "social_security_start_age": w_social_security_start_age.value,
        "pension_monthly": w_pension_monthly.value * 1_000 if w_has_pension.value else 0,
        "pension_start_age": w_pension_start_age.value if w_has_pension.value else None,
        "target_retirement_age": w_target_retirement_age.value,
        "inflation_rate": w_inflation_rate.value / 100,
    }

In [9]:
# --- Output widgets ---
output_stats          = widgets.Output(layout=Layout(width="100%"))
output_projection_plot = widgets.Output(layout=Layout(width="100%"))
output_earliest_age   = widgets.Output(layout=Layout(width="100%"))
output_mc_stats       = widgets.Output(layout=Layout(width="100%"))
output_mc_histogram   = widgets.Output(layout=Layout(width="100%"))
output_sweep_plot     = widgets.Output(layout=Layout(width="100%"))
output_sweep_table    = widgets.Output(layout=Layout(width="100%"))
output_grid_plot      = widgets.Output(layout=Layout(width="100%"))

# --- Sidebar ---
sidebar = widgets.VBox([
    widgets.HTML("<b>Personal</b>"),
    w_current_age,
    w_life_expectancy,
    w_target_retirement_age,
    widgets.HTML("<b>Savings</b>"),
    w_current_savings,
    w_annual_contribution,
    w_contribution_growth_rate,
    widgets.HTML("<b>Returns (nominal)</b>"),
    w_inflation_rate,
    w_pre_retirement_return,
    w_post_retirement_return,
    w_return_volatility,
    widgets.HTML("<b>Spending / Social Security</b>"),
    w_annual_expenses,
    w_social_security_monthly,
    w_social_security_start_age,
    widgets.HTML("<b>Pension</b>"),
    w_has_pension,
    pension_box,
    widgets.HTML("<b>Monte Carlo</b>"),
    w_n_sims,
    run_mc_button,
    run_grid_button,
], layout=Layout(min_width="300px", max_width="300px", overflow_y="auto", padding="8px"))

# --- Main area ---
main_area = widgets.VBox([
    widgets.HTML("<h3>Deterministic Projection</h3>"),
    output_stats,
    output_projection_plot,
    widgets.HTML("<h3>Earliest Retirement Age</h3>"),
    output_earliest_age,
    widgets.HTML("<h3>Monte Carlo Results</h3>"),
    output_mc_stats,
    output_mc_histogram,
    widgets.HTML("<h3>Success Rate vs. Retirement Age</h3>"),
    output_sweep_plot,
    output_sweep_table,
    widgets.HTML("<h3>Success Rate: Age × Balance</h3>"),
    output_grid_plot,
], layout=Layout(flex="1", overflow_y="auto", padding="8px"))

app = widgets.HBox(
    [sidebar, main_area],
    layout=Layout(width="100%", height="100vh", align_items="flex-start")
)
display(app)

AppLayout(children=(VBox(children=(HTML(value='<b>Personal</b>'), BoundedIntText(value=1977, description='Birt…

In [10]:
all_det_widgets = [
    w_current_age, w_life_expectancy, w_current_savings, w_annual_contribution,
    w_contribution_growth_rate, w_inflation_rate, w_pre_retirement_return, w_post_retirement_return,
    w_return_volatility, w_annual_expenses, w_social_security_monthly,
    w_social_security_start_age, w_has_pension, w_pension_monthly, w_pension_start_age,
    w_target_retirement_age,
]


def update_deterministic(change=None):
    a = build_assumptions_from_widgets()
    proj = project_portfolio(a, a["target_retirement_age"])
    depleted = proj.loc[proj["balance_end"] < 0, "age"]
    retire_idx = proj["retired"].idxmax()
    balance_at_retirement = (
        proj.loc[retire_idx - 1, "balance_end"] if retire_idx > 0 else a["current_savings"]
    )

    with output_stats:
        output_stats.clear_output(wait=True)
        print(f"Retirement age tested: {a['target_retirement_age']}")
        print(f"Projected balance at retirement: ${balance_at_retirement/1e3:,.1f}k")
        print(f"Projected balance at life expectancy: ${proj['balance_end'].iloc[-1]/1e3:,.1f}k")
        if not depleted.empty:
            print(f"WARNING: Portfolio runs out at age {int(depleted.iloc[0])}")
        else:
            print("Portfolio lasts through your full life expectancy.")

    with output_projection_plot:
        output_projection_plot.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot(proj["age"] + 1, proj["balance_end"], label="Portfolio balance")
        ax.axvline(a["target_retirement_age"], color="gray", linestyle="--",
                   label=f"Retire at {a['target_retirement_age']}")
        ax.axhline(0, color="red", linewidth=0.8)
        ax.set_xlabel("Age")
        ax.set_ylabel("Balance ($k, today's dollars)")
        ax.set_title("Projected Portfolio Balance")
        ax.legend()
        ax.grid(True)
        ax.yaxis.set_major_formatter(lambda x, _: f"${x/1e3:,.0f}k")
        plt.tight_layout()
        plt.show()
        plt.close("all")

    with output_earliest_age:
        output_earliest_age.clear_output(wait=True)
        earliest, earliest_df = find_earliest_retirement_age(a)
        if earliest is not None:
            print(f"Earliest sustainable retirement age (average scenario): {earliest}")
            print(f"Projected balance at life expectancy: ${earliest_df['balance_end'].iloc[-1]/1e3:,.1f}k")
        else:
            print("No retirement age in the tested range fully sustains your spending.")
            print("Consider increasing savings/contributions, lowering expenses, or extending the age range.")


def run_monte_carlo(btn=None):
    run_mc_button.disabled = True
    run_mc_button.description = "Running..."
    try:
        a = build_assumptions_from_widgets()
        rate, balances = monte_carlo_success(a, a["target_retirement_age"], n_sims=w_n_sims.value)

        with output_mc_stats:
            output_mc_stats.clear_output(wait=True)
            print(f"Retirement age tested: {a['target_retirement_age']}")
            print(f"Success rate (portfolio never depleted): {rate:.1%}")
            print(f"Median ending balance:          ${np.median(balances)/1e3:,.1f}k")
            print(f"10th percentile ending balance: ${np.percentile(balances, 10)/1e3:,.1f}k")
            print(f"90th percentile ending balance: ${np.percentile(balances, 90)/1e3:,.1f}k")

        with output_mc_histogram:
            output_mc_histogram.clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(9, 4))
            ax.hist(balances, bins=40)
            ax.axvline(0, color="red", linewidth=1, label="Depleted")
            ax.set_xlabel("Ending balance at life expectancy ($k, today's dollars)")
            ax.set_ylabel("Number of simulations")
            ax.set_title(f"Distribution of Outcomes — Retiring at {a['target_retirement_age']}")
            ax.xaxis.set_major_formatter(lambda x, _: f"${x/1e3:,.0f}k")
            ax.legend()
            ax.grid(True)
            plt.tight_layout()
            plt.show()
            plt.close("all")

        # Age sweep — capped at 500 sims/age for speed
        sweep_start = max(a["current_age"] + 1, 50)
        sweep_end = min(a["life_expectancy"], 81)
        sweep_results = []
        for age in range(sweep_start, sweep_end):
            r, _ = monte_carlo_success(a, age, n_sims=500)
            sweep_results.append({"retirement_age": age, "success_rate": r})
        sweep_df = pd.DataFrame(sweep_results)

        with output_sweep_plot:
            output_sweep_plot.clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(9, 4))
            ax.plot(sweep_df["retirement_age"], sweep_df["success_rate"] * 100, marker="o")
            ax.axhline(90, color="green", linestyle="--", label="90% success")
            ax.axhline(80, color="orange", linestyle="--", label="80% success")
            ax.set_xlabel("Retirement age")
            ax.set_ylabel("Success rate (%)")
            ax.set_title("Plan Success Rate vs. Retirement Age")
            ax.legend()
            ax.grid(True)
            plt.tight_layout()
            plt.show()
            plt.close("all")

        with output_sweep_table:
            output_sweep_table.clear_output(wait=True)
            display(sweep_df.style.format({"success_rate": "{:.1%}"}))

    finally:
        run_mc_button.disabled = False
        run_mc_button.description = "Run Monte Carlo"


run_mc_button.on_click(run_monte_carlo)

for w in all_det_widgets:
    w.observe(update_deterministic, names="value")


def run_balance_grid(btn=None):
    run_grid_button.disabled = True
    run_grid_button.description = "Running..."
    try:
        a = build_assumptions_from_widgets()
        ages = list(range(max(a["current_age"] + 1, 50), min(a["life_expectancy"], 81), 2))
        # Y-axis in nominal dollars — user compares to their actual account balance.
        low  = a["current_savings"] * 0.25
        high = a["current_savings"] * 8.0
        balances = list(np.linspace(low, high, 12))

        grid = mc_success_grid(a, ages, balances, n_sims=500,
                               inflation_rate=a["inflation_rate"],
                               base_age=a["current_age"])

        with output_grid_plot:
            output_grid_plot.clear_output(wait=True)
            ages_arr = np.array(ages)
            bals_arr = np.array(balances) / 1e3  # nominal $k
            fig, ax = plt.subplots(figsize=(9, 5))
            cf = ax.contourf(ages_arr, bals_arr, grid * 100,
                             levels=np.linspace(0, 100, 21),
                             cmap="RdYlGn", vmin=0, vmax=100)
            plt.colorbar(cf, ax=ax, label="Success rate (%)")
            cs = ax.contour(ages_arr, bals_arr, grid * 100,
                            levels=[80, 90], colors=["black", "black"], linewidths=2)
            ax.clabel(cs, fmt="%d%%", fontsize=9)
            ax.set_xlabel("Retirement age")
            ax.set_ylabel("Portfolio balance at retirement (nominal \$k)")
            ax.set_title("Monte Carlo success rate: retirement age \u00d7 starting balance (nominal dollars)")
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
            plt.close("all")
    finally:
        run_grid_button.disabled = False
        run_grid_button.description = "Run Balance Grid"

run_grid_button.on_click(run_balance_grid)
update_deterministic()  # initial render on load

## Notes & Caveats

This model is intentionally simple so you can read, trust, and extend every
line of it. Things it does **not** account for that you may want to add:

- **Taxes.** Withdrawals from traditional 401(k)/IRA accounts are taxable;
  Roth withdrawals generally aren't. This model treats `annual_expenses` as
  the amount you need *after* tax. For a more accurate picture, separate
  your accounts by tax treatment and model withdrawals/taxes per account.
- **Multiple accounts with different growth rates.** Currently all savings
  are lumped into one balance with one return. You could extend
  `project_portfolio` to track several buckets (e.g. taxable, traditional,
  Roth) with different return assumptions and a withdrawal order.
- **Healthcare costs and Medicare.** Healthcare spending often rises faster
  than general inflation, especially before Medicare eligibility (65).
  Consider modeling a separate, higher-growth expense category.
- **Variable spending in retirement.** Many retirees spend more in early
  "go-go" years and less later. You could make `annual_expenses` a function
  of age instead of a constant.
- **Sequence-of-returns risk near retirement** is partially captured by the
  Monte Carlo simulation, but you could weight early-retirement years more
  heavily or use historical bootstrap sampling instead of a normal
  distribution for more realistic "fat tails."
- **One-time events** (downsizing a home, inheritance, large purchases) can
  be added as one-off adjustments to `balance` in specific years.

### Suggested next steps

- Adjust `target_retirement_age` in the sidebar and watch the projection
  update live.
- Adjust `return_volatility` up or down to see how sensitive your plan is
  to market risk.
- Save copies of this notebook with different assumption sets (e.g.
  `retirement_plan_conservative.ipynb`, `retirement_plan_optimistic.ipynb`)
  to compare scenarios over time.